In [461]:
# DATASET
import json
import os

import numpy as np
import torchaudio
from scipy.io import wavfile
from torch.utils.data import Dataset, DataLoader


class VitsAudioDataset(Dataset):
    def __init__(self, data: list[tuple[str, str]], tokenizer, mel_extractor, sample_rate=16000):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer
        self.sample_rate = sample_rate
        self.mel_extractor = mel_extractor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, id):
        audio_path, text = self.data[id]
        sr, waveform_np = wavfile.read(audio_path)

        # Convert numpy array to PyTorch tensor
        waveform = torch.from_numpy(waveform_np).float().to(device)

        # If the audio is stereo (shape: [samples, 2]), convert it to mono
        if waveform.dim() > 1:
            waveform = waveform.mean(dim=1)

        # Normalize 16-bit PCM audio to [-1.0, 1.0]
        if waveform_np.dtype == np.int16:
            waveform = waveform / 32768.0

        # SciPy shape is [Time, Channels] for stereo, [Time] for mono.
        if waveform.dim() > 1 and waveform.shape[1] > 1:
            waveform = waveform.mean(dim=1)  # Average channels

        if sr != self.sample_rate:
            waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

        if waveform_np.shape[-1] < min_audio_length:
            pad_amount = min_audio_length - waveform_np.shape[-1]
            waveform = F.pad(waveform, (0, pad_amount))

        waveform = waveform.unsqueeze(0)
        mel_spectrogram = self.mel_extractor(waveform).squeeze(0)

        encoding = self.tokenizer(text, padding="max_length", max_length=512, truncation=True, return_tensors="pt")

        input_ids_tensor = encoding["input_ids"].squeeze(0)

        return {
            "audio": waveform.squeeze(0),
            "mel_spectrogram": mel_spectrogram.to(device),
            "input_ids": input_ids_tensor.detach().clone().to(device),
            "attention_mask": encoding["attention_mask"].squeeze(0).detach().clone().to(device),
            "audio_lengths": waveform.shape[-1],
            "text_lengths": (input_ids_tensor != self.tokenizer.pad_token_id).sum().item()
        }


In [462]:
# COLLATE
import torch.nn.utils.rnn as rnn_utils


def vits_collate_fn(batch):
    # 1. Separate the batch into lists
    audios = [item["audio"] for item in batch]
    mel_specs = [item["mel_spectrogram"] for item in batch]
    input_ids = [item["input_ids"] for item in batch]

    audio_lens = torch.tensor([item["audio_lengths"] for item in batch], dtype=torch.long)
    text_lens = torch.tensor([item["text_lengths"] for item in batch], dtype=torch.long)

    # 2. Pad Audio (pad with 0.0)
    # Shape: [batch_size, max_audio_len]
    audio_padded = rnn_utils.pad_sequence(audios, batch_first=True, padding_value=0.0)

    # 3. Pad Mel Spectrograms (pad with 0.0)
    # Shape: [batch_size, n_mels, max_mel_len]
    # Note: pad_sequence expects [seq_len, ...], so we transpose, pad, and transpose back
    mel_specs_transposed = [m.transpose(0, 1) for m in mel_specs]
    mel_padded_transposed = rnn_utils.pad_sequence(mel_specs_transposed, batch_first=True, padding_value=0.0)
    mel_padded = mel_padded_transposed.transpose(1, 2)  # Back to [batch, n_mels, time]

    # 4. Pad Input IDs (pad with tokenizer's pad_token_id, usually 0)
    input_ids_padded = rnn_utils.pad_sequence(input_ids, batch_first=True, padding_value=0)

    return {
        "audio": audio_padded,
        "mel_spectrogram": mel_padded,
        "input_ids": input_ids_padded,
        "audio_lengths": audio_lens,
        "text_lengths": text_lens
    }

In [463]:
# DATALOADER
def create_data_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=vits_collate_fn,
        num_workers=0,
        pin_memory=False
    )

In [464]:
# ALIGNED DURATIONS
import torch


def get_aligned_durations(text_len, mel_len, batch_size, device):
    """
    Creates durations that sum EXACTLY to mel_len.
    This guarantees the upsampled tensor perfectly matches the target mel length.
    """
    base_dur = mel_len // text_len
    remainder = mel_len % text_len

    # Start with base duration for all tokens
    durs = torch.full((batch_size, text_len), base_dur, device=device, dtype=torch.long)

    # Distribute the remainder to the first few tokens to make the sum exact
    for b in range(batch_size):
        durs[b, :remainder] += 1

    return durs

In [465]:
# GENERATORS

import torch.nn as nn


class HiFiGANGenerator(nn.Module):
    def __init__(
            self,
            initial_channel,
            resblock_kernel_sizes,
            resblock_dilation_sizes,
            upsample_rates,
            upsample_initial_channel,
            upsample_kernel_sizes,
    ):
        super().__init__()
        self.num_kernels = len(resblock_kernel_sizes)
        self.num_upsamples = len(upsample_kernel_sizes)

        self.conv_pre = nn.Conv1d(initial_channel, upsample_initial_channel, 7, 1, padding=3)

        self.ups = nn.ModuleList()
        for i, (u, k) in enumerate(zip(upsample_rates, upsample_kernel_sizes)):
            self.ups.append(
                nn.ConvTranspose1d(
                    upsample_initial_channel // (2 ** i),
                    upsample_initial_channel // (2 ** (i + 1)),
                    k,
                    u,
                    padding=(k - u) // 2
                )
            )
        self.resblocks = nn.ModuleList()

        ch = upsample_initial_channel // 2
        for i in range(len(self.ups)):
            ch = upsample_initial_channel // (2 ** (i + 1))
            resblocks_list = nn.ModuleList()
            for k, d in zip(resblock_kernel_sizes, resblock_dilation_sizes):
                resblocks_list.append(nn.Conv1d(ch, ch, k, 1, dilation=d, padding=(k * d - d) // 2))
            self.resblocks.append(resblocks_list)

        self.conv_post = nn.Conv1d(ch, 1, 7, 1, padding=3)

    def forward(self, input):
        value = self.conv_pre(input)
        for i in range(self.num_upsamples):
            value = F.leaky_relu(value, 0.1)
            value = self.ups[i](value)
            xs = 0
            for j in range(self.num_kernels):
                xs += F.leaky_relu(self.resblocks[i][j](value), 0.1)
            value = xs / self.num_kernels
        value = F.leaky_relu(value)
        value = torch.tanh(self.conv_post(value))

        return value


class SimplifiedVITSGenerator(nn.Module):
    def __init__(self, vocab_size, hidden_channels=192):
        super().__init__()
        self.hidden_channels = hidden_channels
        assert hidden_channels != vocab_size

        # 1. Text encoder
        self.emb = nn.Embedding(vocab_size, hidden_channels)
        self.encoder = nn.Sequential(
            nn.Conv1d(hidden_channels, hidden_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, hidden_channels, 3, padding=1),
            nn.ReLU(),
        )

        # 2. Duration Predictor (Predicts how long each phoneme lasts)
        self.duration_predictor = nn.Sequential(
            nn.Conv1d(hidden_channels, hidden_channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_channels, 1, 3, padding=1),
        )

        # 3. HiFi-GAN Decoder (Upsamples to 16kHz audio)
        self.decoder = HiFiGANGenerator(
            initial_channel=hidden_channels,
            resblock_kernel_sizes=[3, 7, 11],
            resblock_dilation_sizes=[1, 3, 5],
            upsample_rates=[8, 8, 2, 2],
            upsample_initial_channel=512,
            upsample_kernel_sizes=[16, 16, 4, 4],
        )

    def forward(self, input_ids, mel_spec_target=None, training=True):
        # 1. Encode Text
        # input_ids shape: [batch, seq_len]
        x = self.emb(input_ids)  # [batch, seq_len, hidden]
        x = x.transpose(1, 2)  # [batch, hidden, seq_len]
        x = self.encoder(x)  # [batch, hidden*2, seq_len]

        log_dur_predict = self.duration_predictor(x).squeeze(1)  # [batch, seq_len]

        b, c, t = x.shape

        if training:
            durs = get_aligned_durations(t, mel_spec_target, b, x.device)
        else:
            durs = torch.round(torch.exp(log_dur_predict)).clamp(min=1).long()
            # durs = get_aligned_durations(t, mel_spec_target, b, x.device)

        expanded_list = []
        for i in range(b):
            exp_b = torch.repeat_interleave(x[i], durs[i], dim=-1)
            expanded_list.append(exp_b)

        x_upsampled = torch.stack(expanded_list)

        # # Split into mean and log variance (simplified VAE)
        # m_p, logs_p = torch.split(x, [self.hidden_channels, self.hidden_channels], dim=1)
        #
        # # # 2. Predict Durations (simplified)
        # # # We cheat slightly and use the target mel length to guide duration predictor
        # # duration_prediction = self.duration_predictor(m_p.detach()).squeeze(1)
        # # duration_prediction = F.softplus(duration_prediction)
        #
        # # 3. Upsample features to match mel length (simplified alignment)
        # batch_size, _, mel_len = mel_spec_target.shape
        # _, _, text_len = m_p.shape
        #
        # # Calculate repeat factor (total mel frames / total text tokens)
        # repeat_factor = max(1, int(mel_len / text_len))
        #
        # # Repeat the features to match the time dimension of the mel spectrogram
        # # Shape: [batch, hidden, mel_len]
        # x_upsampled = m_p.repeat_interleave(repeat_factor, dim=-1)
        # x_upsampled = x_upsampled[:, :, :mel_len]  # Trim to exact mel length

        # 4. Decode to Audio
        audio_generated = self.decoder(x_upsampled)

        # kl_loss = -0.5 * torch.sum(1 + logs_p - m_p.pow(2) - logs_p.exp()) / (batch_size * text_len)

        log_durs = torch.log(durs.float() + 1e+8)
        dur_loss = F.mse_loss(log_durs, log_dur_predict)

        return {
            "audio_generated": audio_generated,
            "dur_loss": dur_loss
        }


In [466]:
# DISCRIMINATORS
import torch.nn as nn
import torch.nn.functional as F


class DiscriminatorP(nn.Module):
    def __init__(self, period, kernel_size=5, stride=3):
        super().__init__()
        self.period = period
        norm_f = nn.utils.parametrizations.weight_norm

        self.convs = nn.ModuleList([
            norm_f(nn.Conv2d(1, 32, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(32, 128, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(128, 512, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(512, 1024, (kernel_size, 1), (stride, 1), padding=(2, 0))),
            norm_f(nn.Conv2d(1024, 1024, (kernel_size, 1), (stride, 1), padding=(2, 0))),
        ])

        self.conv_post = norm_f(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, input, return_features=False):
        x = input
        fmap = []

        # Reshape 1D audio [B, 1, T] to 2D [B, 1, T//period, period]
        b, c, t = input.shape

        if t % self.period != 0:
            pad = self.period - (t % self.period)
            x = F.pad(x, (0, pad), "reflect")
            t = t + pad
        x = x.view(b, c, -1, self.period)

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, 0.1)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)

        if return_features:
            return x, fmap

        return x


class MultiperiodDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([
            DiscriminatorP(2),
            DiscriminatorP(3),
            DiscriminatorP(5),
            DiscriminatorP(7),
            DiscriminatorP(11),
        ])

    def forward(self, y, return_features=False):
        y_ds = []
        fmaps = []

        for i, d in enumerate(self.discriminators):
            if return_features:
                y_d, fmap = d(y, return_features=True)
            else:
                y_d = d(y)
                fmap = []

            y_ds.append(y_d)
            fmaps.append(fmap)

        return y_ds, fmaps



In [467]:
# class SimpleGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.enc = nn.Sequential(
#             nn.Conv1d(1, 32, 15, stride=2, padding=7),
#             nn.ReLU(),
#             nn.Conv1d(32, 64, 15, stride=2, padding=7),
#             nn.ReLU(),
#             nn.Conv1d(64, 128, 15, stride=2, padding=7),
#             nn.ReLU(),
#         )
#
#         self.dec = nn.Sequential(
#             nn.ConvTranspose1d(128, 64, 16, stride=2, padding=7),
#             nn.ReLU(),
#             nn.ConvTranspose1d(64, 32, 16, stride=2, padding=7),
#             nn.ReLU(),
#             nn.ConvTranspose1d(32, 1, 16, stride=2, padding=7),
#             nn.Tanh(),
#         )
#
#     def forward(self, input):
#         z = self.enc(input)
#         y_hat = self.dec(z)
#         return y_hat[:, :, :input.shape[-1]]
#
#
# class SimpleDiscriminator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Conv1d(1, 32, 15, stride=2, padding=7),
#             nn.LeakyReLU(0.2),
#             nn.Conv1d(32, 64, 15, stride=2, padding=7),
#             nn.LeakyReLU(0.2),
#             nn.Conv1d(64, 128, 15, stride=2, padding=7),
#             nn.LeakyReLU(0.2),
#             nn.AdaptiveAvgPool1d(1),
#             nn.Flatten(),
#             nn.Linear(128, 1),
#         )
#
#     def forward(self, input, return_features=False):
#         fmap = []
#         out = input
#         for layer in self.layers[:-2]:
#             out = layer(out)
#             if return_features:
#                 fmap.append(out)
#         out = self.layers[-2:](out)
#         if return_features:
#             return out, fmap
#         return out


In [468]:
class SimpleVITS(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.generator = SimplifiedVITSGenerator(vocab_size, hidden_channels=192)
        self.discriminator = MultiperiodDiscriminator()



In [469]:
# TRAIN STEP


def train_step(batch, model: SimpleVITS, optimizer_g, optimizer_d, device, mel_extractor):
    audio = batch["audio"].to(device)
    mel_spectrogram = batch["mel_spectrogram"].to(device)
    input_ids = batch["input_ids"].to(device)

    segment_size = 8192  # Or 16384 depending on your config
    if audio.size(-1) > segment_size:
        # Randomly pick a starting point
        max_audio_start = audio.size(-1) - segment_size
        audio_start = torch.randint(0, max_audio_start, (1,)).item()

        # Slice audio and corresponding mel spectrogram
        # Note: mel spectrogram hop_length is usually 256, so we divide audio_start by 256
        audio = audio[:, audio_start: audio_start + segment_size]
        mel_start = audio_start // 256
        mel_spectrogram = mel_spectrogram[:, :, mel_start: mel_start + (segment_size // 256)]
    elif audio.size(-1) < segment_size:
        # If too short, pad with zeros to reach exactly 8192
        pad_len = segment_size - audio.size(-1)
        audio = F.pad(audio, (0, pad_len))
        mel_spectrogram = F.pad(mel_spectrogram, (0, pad_len // 256))

    mel_len_target = max(mel_spectrogram.shape[-1], 2)

    # Generator pass
    output = model.generator(input_ids=input_ids, mel_spec_target=mel_len_target)

    y_hat = output["audio_generated"]
    dur_loss = output["dur_loss"]

    if y_hat.size(-1) > audio.size(-1):
        y_hat = y_hat[:, :, :audio.size(-1)]
    elif y_hat.size(-1) < audio.size(-1):
        y_hat = F.pad(y_hat, (0, audio.size(-1) - y_hat.size(-1)))

    mel_hat = mel_extractor(y_hat).squeeze(1)

    if mel_hat.size(-1) > mel_spectrogram.size(-1):
        mel_hat = mel_hat[:, :, :mel_spectrogram.size(-1)]
    elif mel_hat.size(-1) < mel_spectrogram.size(-1):
        mel_hat = F.pad(mel_hat, (0, mel_spectrogram.size(-1) - mel_hat.size(-1)))

    mel_loss = nn.L1Loss()(mel_hat, mel_spectrogram)
    total_g_loss = mel_loss + dur_loss * 2.0

    # Discriminator update
    optimizer_d.zero_grad()
    y_hat_detached = y_hat.detach()

    d_real, _ = model.discriminator(audio.unsqueeze(1) if audio.dim() == 2 else audio)
    d_fake, _ = model.discriminator(y_hat_detached)

    ## Hinge loss
    loss_d_real = sum(torch.mean(torch.relu(0.9 - d)) for d in d_real) / len(d_real)
    loss_d_fake = sum(torch.mean(torch.relu(1.0 + d)) for d in d_fake) / len(d_fake)
    total_d_loss = loss_d_real + loss_d_fake

    # Backprop
    total_d_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.discriminator.parameters(), max_norm=1.0)
    optimizer_d.step()

    # Generator update
    optimizer_g.zero_grad()

    _, f_real = model.discriminator(audio.unsqueeze(1) if audio.dim() == 2 else audio,
                                    return_features=True)
    d_fake, f_fake = model.discriminator(y_hat,
                                         return_features=True)

    ## Adversarial loss. Need to have higher values on fake values
    ## Discriminator tries to push d_fake to -1, so adv_loss should be 1
    # so that generator learns how to fool discriminator
    adv_loss = -sum(torch.mean(d) for d in d_fake) / len(d_fake)

    # Feature-matching loss
    fm_losses = []
    for drs, dfs in zip(f_real, f_fake):
        fm_loss = 0
        for dr, df in zip(drs, dfs):
            fm_loss += nn.L1Loss()(dr, df)
        fm_loss = fm_loss / len(drs)
        fm_losses.append(fm_loss)

    final_g_loss = total_g_loss + adv_loss + sum(fm_losses) / len(fm_losses) * 2

    # Backprop
    final_g_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.generator.parameters(), max_norm=1.0)
    optimizer_g.step()

    if torch.isnan(final_g_loss) or torch.isnan(total_d_loss):
        print("⚠️ NaN detected! Skipping batch to save model weights.")
        optimizer_g.zero_grad()
        optimizer_d.zero_grad()
        return {"g_loss": 0.0, "d_loss": 0.0, "mel_loss": 0.0, "dur_loss": 0.0}

    return {
        "g_loss": final_g_loss.item(),
        "d_loss": total_d_loss.item(),
        "mel_loss": mel_loss.item(),
        "dur_loss": dur_loss.item(),
    }


In [470]:
from tqdm import tqdm
from transformers import VitsTokenizer

tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-rus")
vocab_size = len(tokenizer)

num_epochs = 100
save_every = 1000
global_step = 0

min_audio_length = 1024

device = "cuda:0" if torch.cuda.is_available() else "cpu"

mel_extractor = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    # number of digital samples analyzed in a single window of time
    n_fft=1024,
    win_length=min_audio_length,
    hop_length=512,
    n_mels=80,
    f_min=0,
    f_max=8000,
).to(device)

model = SimpleVITS(vocab_size).to(device)

# Separate the optimizers as your code requires
optimizer_g = torch.optim.AdamW(model.generator.parameters(), lr=5e-5, betas=(0.8, 0.99))
optimizer_d = torch.optim.AdamW(model.discriminator.parameters(), lr=5e-5, betas=(0.8, 0.99))

model.train()

with open("sonya_dataset/index.json", 'r', encoding='utf-8') as f:
    index = json.load(f)
    data = [(audio_path, index[audio_path]) for audio_path in index]

dataset = VitsAudioDataset(data, tokenizer, mel_extractor)
data_loader = create_data_loader(dataset)

if not os.path.exists("./checkpoint"):
    os.mkdir("./checkpoints")

for epoch in range(num_epochs):
    progres_bar = tqdm(data_loader, desc="Epoch {}".format(epoch + 1))
    for batch in progres_bar:
        losses = train_step(batch, model, optimizer_g, optimizer_d, device, mel_extractor)

        progres_bar.set_postfix({
            "G_loss": f"{losses['g_loss']:.4f}",
            "D_loss": f"{losses['d_loss']:.4f}",
            "Mel": f"{losses['mel_loss']:.4f}",
            "dur_loss": f"{losses['dur_loss']:.4f}",
        })

        global_step += 1

        if global_step % save_every == 0:
            checkpoint = {
                "step": global_step,
                "generator": model.generator.state_dict(),
                "discriminator": model.discriminator.state_dict(),
                "optimizer_g": optimizer_g,
                "optimizer_d": optimizer_d,
            }

            torch.save(checkpoint, f"./checkpoints/checkpoint-{global_step}.pth")
            print(f"\n✅ Saved checkpoint at step {global_step}")

Epoch 16:  43%|████▎     | 28/65 [00:01<00:03, 10.53it/s, G_loss=nan, D_loss=2.0206, Mel=nan, dur_loss=0.1776]        


✅ Saved checkpoint at step 1000


Epoch 31:  80%|████████  | 52/65 [00:03<00:01,  8.57it/s, G_loss=42.7123, D_loss=0.6032, Mel=37.2045, dur_loss=0.0156]


✅ Saved checkpoint at step 2000


Epoch 47:  18%|█▊        | 12/65 [00:01<00:07,  6.93it/s, G_loss=9.7419, D_loss=0.0740, Mel=1.2449, dur_loss=0.2537]  

...

Epoch 77:  95%|█████████▌| 62/65 [00:04<00:00, 10.29it/s, G_loss=nan, D_loss=2.0002, Mel=nan, dur_loss=0.1575]        


✅ Saved checkpoint at step 5000


Epoch 93:  34%|███▍      | 22/65 [00:01<00:04, 10.09it/s, G_loss=nan, D_loss=1.0194, Mel=nan, dur_loss=0.5815]        


✅ Saved checkpoint at step 6000


Epoch 100: 100%|██████████| 65/65 [00:03<00:00, 18.43it/s, G_loss=nan, D_loss=0.0799, Mel=nan, dur_loss=0.1103]        
